In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

# Frame-Level Metric Analysis
Advanced visualization with frame-level aggregation and filtering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Callable, Optional, Union
from pathlib import Path

from klarity import geometry
from klarity import metrics
from klarity import parsing
from klarity import viz

# Type hint for reducer functions
Reducer = Union[str, Callable[[pd.Series], float]]

## Configuration

In [ ]:
import config

OUTPUT_DIR = config.HEATMAPS_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

PLACEMENTS = [
    "placement_1",
    "placement_2",
    "placement_3",
    "placement_4",
    "placement_5",
    "placement_6",
]

XANTHAN_LEVELS = ["000 xanthan", "0125 xanthan", "025 xanthan"]

RPM_LEVELS = [75, 100, 125, 150]
AER_LEVELS = [45, 55, 70, 90]

In [ ]:
from klarity import io

io.check_dataframes_stale()

In [ ]:
# Display labels for plots
AXIS_LABELS = {
    "x": r"P V$^{-1}$ [W L$^{-1}$]",
    "y": r"$\dot{\mathrm{V}}_{\mathrm{gas}}$ [L min$^{-1}$]",
    "x2": r"$\mu_\mathrm{app}$ [mPa s]",
}

XANTHAN_LABELS = {
    "000 xanthan": "0.000 wt% xanthan",
    "0125 xanthan": "0.125 wt% xanthan",
    "025 xanthan": "0.250 wt% xanthan",
}

PLACEMENT_LABELS = {
    "placement_1": "Position 1",
    "placement_2": "Position 2",
    "placement_3": "Position 3",
    "placement_4": "Position 4",
    "placement_5": "Position 5",
    "placement_6": "Position 6",
}

# P/V values [W L-1] per xanthan level and RPM setting
PV_MAP = {
    "000 xanthan":  {75: 0.52, 100: 0.85, 125: 1.42, 150: 2.24},
    "0125 xanthan": {75: 0.28, 100: 0.57, 125: 0.95, 150: 1.57},
    "025 xanthan":  {75: 0.22, 100: 0.49, 125: 0.89, 150: 1.48},
}

# Apparent viscosity at each RPM setting (mPa·s), derived from rheology fits.
VISCOSITY_MAP = {
    "000 xanthan":  {75: 1,   100: 1,   125: 1,   150: 1},
    "0125 xanthan": {75: 31,  100: 21,  125: 16,  150: 13},
    "025 xanthan":  {75: 168, 100: 104, 125: 71,  150: 52},
}


## Load Bubble-Level Data

Load pre-computed bubble-level data

In [ ]:
bubble_df = pd.read_pickle(config.BUBBLE_LEVEL_PKL)
print(f"✓ Loaded {len(bubble_df):,} bubbles")

## Compute Setting-Level Aggregates

For direct aggregation to (placement, reactor_setting) level

In [ ]:
print("Computing setting-level aggregates...")

acc = metrics.collect_setting_accumulators(
    bubble_df,
    placement_level="placement",
    setting_level="reactor_setting",
    geom_col="model_used",
    volume_col="bubble_volume_mm3",
    surface_col="bubble_surface_area_mm2",
    diameter_col="equivalent_diameter_mm",
)

print("✓ Accumulators computed")

In [ ]:
print("Creating aggregate dataframe...")

agg = metrics.accumulators_to_frame(
    acc,
    placement_name="placement",
    setting_name="reactor_setting",
)

# Add rpm/aeration/xanthan columns
agg = metrics.enrich_with_setting_info(agg, setting_col="reactor_setting")

# Add observed volume metrics
agg = metrics.add_observed_volume_metrics(
    agg,
    geometry_module=geometry,
)

print(f"✓ Aggregate data ready: {len(agg)} setting combinations")

## Load Frame-Level Data

In [ ]:
frame_df = pd.read_pickle(config.FRAME_LEVEL_PKL)
print(f"Loaded {len(frame_df):,} frames")

## Filter Data (Optional)

Remove problematic settings or outliers if needed

In [ ]:
# Option 1: Use all data
frame_df_filtered = frame_df.copy()

# Option 2: Filter by RPM/aeration
# frame_df_filtered = frame_df[
#     (frame_df['rpm_val'].isin(RPM_LEVELS)) &
#     (frame_df['aer_val'].isin(AER_LEVELS))
# ]

# Option 3: Remove specific problematic settings
# problematic_settings = ['150 rpm 90 lmin 000 xanthan']  # Example
# frame_df_filtered = frame_df[~frame_df['reactor_setting'].isin(problematic_settings)]

# Option 4: Filter by placement
# frame_df_filtered = frame_df[frame_df['placement'].isin(PLACEMENTS)]

print(f"Frames after filtering: {len(frame_df_filtered):,} (from {len(frame_df):,})")

## Remove Extreme Outliers (Optional)

For cleaner visualizations, you can remove extreme outliers

In [ ]:
# Example: Remove frames with unreasonably high gas holdup
# (adjust threshold based on your system)
# frame_df_trimmed = frame_df_filtered[frame_df_filtered['epsilon_obs'] < 0.5]

# Or keep all data
frame_df_trimmed = frame_df_filtered.copy()

print(f"Frames after trimming: {len(frame_df_trimmed):,}")

## Create Visualizations

Generate heatmaps from frame-level data with different aggregation strategies

### Gas Holdup Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"All visualizations saved to: {OUTPUT_DIR}")
print(f"{'='*60}")
print(f"\nFiles created:")
for pdf in sorted(OUTPUT_DIR.glob("*.pdf")):
    print(f"  - {pdf.name}")

In [ ]:
SHARED_PLOT_KWARGS = dict(
    placements=PLACEMENTS,
    xanthan_order=XANTHAN_LEVELS,
    axis_label_map=AXIS_LABELS,
    xanthan_label_map=XANTHAN_LABELS,
    placement_label_map=PLACEMENT_LABELS,
    placements_keep=PLACEMENTS,
    xanthan_levels=XANTHAN_LEVELS,
    rpm_levels_keep=RPM_LEVELS,
    aer_levels_keep=AER_LEVELS,
    viscosity_map=VISCOSITY_MAP,
    viscosity_label=AXIS_LABELS["x2"],
    viscosity_decimals=0,
    viscosity_label_pad=10,
    pv_map=PV_MAP,
    dpi=1200,
)

In [ ]:
for metric_col, spec in viz.METRIC_SPECS.items():
    agg, mcol = metrics.aggregate_frames_for_grid(frame_df_trimmed, metric_col, reducer="mean")
    viz.plot_metric_grid_from_agg(
        agg,
        metric_col=mcol,
        title=spec["title"],
        colorbar_label=spec["cbar"],
        robust=spec["robust"],
        vmin=spec["vmin"],
        vmax=spec["vmax"],
        outpath=OUTPUT_DIR / f"{metric_col}.pdf",
        annotate_cells=spec.get("annotate", True),
        annotation_decimals=spec.get("annotation_decimals", 2),
        **SHARED_PLOT_KWARGS,
    )
    plt.show()